# 01 | Data Cleaning: IEA RDDPUBLIC Panel Construction

This notebook loads the IEA Energy Technology RD&D Budgets public-sector dataset
(RDDPUBLIC), filters it to a clean USD-PPP-constant panel, drops IEA aggregates,
and collapses the 4-digit technology codes to 9 two-digit categories. Output is
a tidy panel with one row per (country, technology, year).


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RAW = "../data/raw/OECD.IEA,RDDPUBLIC,1.0,filtered,2026-05-04 00-07-52.csv"


## Chunked load — the raw IEA file is ~257 MB

In [ ]:
# Read in chunks and apply filters as we go
chunks = []
for chunk in pd.read_csv(RAW, chunksize=200_000, low_memory=False):
    keep = (
        (chunk["UNIT_MEASURE"] == "USD")
        & (chunk["PRICE_BASE"] == "V_PPP")
        & (chunk["TRANSACTION"] == "RDD")
        & (chunk["FUNDS"] == "TOTAL_PUB")
        & (chunk["CONFIDENTIALITY_STATUS"] != "C")
    )
    chunks.append(chunk.loc[keep])

raw = pd.concat(chunks, ignore_index=True)
print(f"rows after filters: {len(raw):,}")


## Drop IEA aggregates and collapse technologies to 9 two-digit categories

In [ ]:
IEA_AGGREGATES = {"IEA_TOTAL", "IEA_OECD", "IEA_NON_OECD", "IEA_NA", "IEA_EUR"}
raw = raw[~raw["REF_AREA"].isin(IEA_AGGREGATES)]

# Tech mapping: keep only 2-digit codes (the 9 high-level categories)
TECH_MAP = {
    "1":  "Energy efficiency",
    "2":  "Fossil fuels",
    "3":  "Renewables",
    "4":  "Nuclear",
    "5":  "Hydrogen & fuel cells",
    "6":  "Other power & storage",
    "7":  "Other crosscutting",
    "8":  "CO2 capture & storage",
    "9":  "Unallocated",
}

# Use the top-level category (group 2-digit codes by their first digit family)
TWO_DIGIT_TO_CAT = {
    "1": "Energy efficiency",
    "2": "Fossil fuels",
    "31": "Solar", "32": "Wind", "33": "Ocean",
    "34": "Biofuels", "35": "Geothermal", "36": "Hydropower",
    "4": "Nuclear",
    "5": "Hydrogen & fuel cells",
    "8": "CO2 capture & storage",
}

def map_tech(code):
    code = str(code)
    for prefix in sorted(TWO_DIGIT_TO_CAT.keys(), key=len, reverse=True):
        if code.startswith(prefix):
            return TWO_DIGIT_TO_CAT[prefix]
    return None

raw["technology"] = raw["TECHNOLOGY"].apply(map_tech)
raw = raw.dropna(subset=["technology"])
print("technologies kept:", sorted(raw["technology"].unique()))


## Collapse to (country, technology, year) panel

In [ ]:
raw = raw.rename(columns={"REF_AREA": "country", "TIME_PERIOD": "year", "OBS_VALUE": "spending_usd_ppp_millions"})
raw["year"] = raw["year"].astype(int)

panel = (
    raw.groupby(["country", "technology", "year"], as_index=False)["spending_usd_ppp_millions"]
       .sum()
)
panel = panel[panel["spending_usd_ppp_millions"] >= 0]
print(f"panel rows: {len(panel):,}; countries: {panel['country'].nunique()}; years: {panel['year'].min()}–{panel['year'].max()}")
panel.head()


## Save processed panel

In [ ]:
panel.to_csv("../data/processed/rdd_public_panel.csv", index=False)
print("saved → ../data/processed/rdd_public_panel.csv")


## Visual check: total spending per technology over time

In [ ]:
totals = panel.groupby(["year", "technology"], as_index=False)["spending_usd_ppp_millions"].sum()

fig, ax = plt.subplots(figsize=(11, 6))
for tech, sub in totals.groupby("technology"):
    ax.plot(sub["year"], sub["spending_usd_ppp_millions"], label=tech, lw=1.5)
ax.set_xlabel("Year")
ax.set_ylabel("USD millions (PPP, constant)")
ax.set_title("Public RD&D spending by technology, OECD aggregate, 1974–2023")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=9)
plt.tight_layout()
plt.savefig("../figures/rdd_public_panel_lineplot.png", dpi=150, bbox_inches="tight")
plt.show()


## Output

- `../data/processed/rdd_public_panel.csv`, clean panel with 4 columns:
  `country`, `technology`, `year`, `spending_usd_ppp_millions`.
- `../figures/rdd_public_panel_lineplot.png`, visual sanity check.

Nuclear dominates the early period; renewables, hydrogen, and CCS expand after 2000.
